In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning import Trainer  # Add this line
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger
import numpy as np

class CryptoDataset(Dataset):
    def __init__(self, features: pd.DataFrame, labels: pd.DataFrame, transform=None):
        self.features = torch.tensor(features.values, dtype=torch.float32)
        self.labels = torch.tensor(LabelEncoder().fit_transform(labels['&-target']), dtype=torch.long)
        self.transform = transform

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        feature = self.features[idx]
        if self.transform:
            feature = self.transform(feature)
        return feature, self.labels[idx]

class CryptoDataModule(pl.LightningDataModule):
    def __init__(self, directory_path, batch_size=128, num_workers=4, train_split=0.8):
        super().__init__()
        self.directory_path = directory_path
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.train_split = train_split
        self.scaler = StandardScaler()
        
    def setup(self, stage=None):
        # Load and preprocess data with new file paths
        features_path = os.path.join(self.directory_path, 'ETH/USDT:USDT_features_20250203_114859.parquet')
        labels_path = os.path.join(self.directory_path, 'ETH/USDT:USDT_labels_20250203_114859.parquet')
        if not (os.path.exists(features_path) and os.path.exists(labels_path)):
            raise FileNotFoundError(f"Data files not found in {self.directory_path}/ETH/")
            
        features = pd.read_parquet(features_path)
        labels = pd.read_parquet(labels_path)
        
        # Scale features
        features_scaled = self.scaler.fit_transform(features)
        features = pd.DataFrame(features_scaled, columns=features.columns)
        
        # Create dataset
        dataset = CryptoDataset(features, labels)
        
        # Split dataset
        train_size = int(self.train_split * len(dataset))
        val_size = len(dataset) - train_size
        self.train_dataset, self.val_dataset = random_split(dataset, [train_size, val_size])
        
        # Store dimensions
        self.input_dim = features.shape[1]
        self.output_dim = len(labels['&-target'].unique())

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, 
                        num_workers=self.num_workers, shuffle=True, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, 
                        num_workers=self.num_workers, pin_memory=True)
